## 1. Filtros

Observaciones:
- Se excluyen: filas vacías (4), filas sin fecha (2) y egresos `otros` + `arreglos local` (18)
- Antes de cargar, se verifica cuántas filas excluye cada filtro

In [0]:
%sql

select 
    count(*) as total_bronze,
    count_if(coalesce(fecha, cantidad, producto, talla, precio_dado, precio_original, ajuste_precio, precio_total, forma_pago, regateo, categoria, observaciones) is null) as filas_vacias,
    count_if(fecha is null or try_to_date(replace(fecha, '-', '/'), 'dd/MM/yyyy') is null) as sin_fecha,
    count_if(lower(trim(categoria)) = 'otros' and lower(trim(producto)) = 'arreglos local') as egresos
from fabriperezchauca.bronze.ventas

## 2. Vista previa de silver

Observaciones:
- Aplica los filtros y todas las transformaciones definidas en el EDA
- Se valida antes de cargar en `silver.ventas`
- Se excluyen las filas sin `cantidad` o sin `precio_venta` (13 filas): no tienen monto y no aportan a ningún KPI de ingresos. Su aporte a unidades vendidas (KPI 2a) es despreciable

In [0]:
%sql
create or replace temp view silver_ventas_preview as

with base as (
    select 
        try_to_date(replace(fecha, '-', '/'), 'dd/MM/yyyy') as f,
        lower(trim(categoria))                              as categoria,
        lower(trim(producto))                               as producto,
        talla,
        try_cast(cantidad as int),
        try_cast(regexp_replace(precio_original, '[^0-9.-]', '') as decimal(10, 2)) as precio_original,
        try_cast(regexp_replace(precio_dado, '[^0-9.-]', '')     as decimal(10, 2)) as precio_venta,
        lower(trim(forma_pago))                                  as forma_pago,
        observaciones
    from fabriperezchauca.bronze.ventas
)

select 
    make_date(2026, month(f), day(f)) as fecha,
    case categoria 
        when 'accesorio'            then 'accesorios'
        when 'acccesorios'          then 'accesorios'
        when 'uniforme'             then 'uniformes'
        when 'unifoemes'            then 'uniformes'
        when 'equipo táctico'       then 'equipo tactico'
        when 'equipamiento tactico' then 'equipo tactico'
        when '<valor exacto>'       then 'equipo tactico'
        ELSE categoria
    end as categoria,
    producto,
    talla,
    cantidad,
    precio_original,
    precio_venta,
    greatest(cantidad * (precio_original - precio_venta), 0) as descuento,
    greatest(cantidad * (precio_venta - precio_original), 0) as recargo,
    cantidad * precio_venta                                  as total,
    case forma_pago
        when 'tarjeta credito' then 'tarjeta de credito'
        else forma_pago
    end as forma_pago,
    observaciones
from base
where f is not null 
    and not (categoria <=> 'otros' and producto <=> 'arreglos local')
    and cantidad is not null
    and precio_venta is not null

In [0]:
%sql
select 
    count(*)                       as filas,
    count_if(cantidad is null)     as cantidad_null,
    count_if(precio_venta is null) as precio_venta_null,
    count(distinct categoria)      as categorias,
    count(distinct forma_pago)     as formas_pago
from silver_ventas_preview

In [0]:
%sql
select *
from silver_ventas_preview
limit 10

## 3. Carga en silver

Observaciones:
- Se cargan las filas de la vista previa ya validada
- `_source_table` y `_processing_timestamp` toman su valor por defecto

In [0]:
%sql

insert overwrite fabriperezchauca.silver.ventas (
    fecha, categoria, producto, talla, cantidad, precio_original, precio_venta, descuento, recargo, total, forma_pago, observaciones
)

select fecha, categoria, producto, talla, cantidad, precio_original, precio_venta, descuento, recargo, total, forma_pago, observaciones
from silver_ventas_preview

In [0]:
%sql
select 
    count(*) as filas,
    count_if(_source_table is null) as sin_metadatos
from fabriperezchauca.silver.ventas

In [0]:
%sql
select * 
from fabriperezchauca.silver.ventas
limit 10